# Notebook 2 — Create the Labels

This notebook creates the target label for the Olist delivery prediction problem.

The goal is to classify each order as:

- `1` = Late delivery
- `0` = On-time delivery

The label is created by comparing the actual customer delivery date
with the estimated delivery date.

The input artifact is the order-level ML table created in Notebook 1.

In [2]:
import pandas as pd 

In [3]:
ml_table = pd.read_csv(
    "artifacts/order_level_ml_table.csv"
)

print("Shape:", ml_table.shape)
print("Columns:")
print(ml_table.columns.tolist())

Shape: (99441, 32)
Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'geolocation_zip_code_prefix', 'latitude', 'longitude', 'city', 'state', 'item_count', 'unique_products', 'unique_sellers', 'total_item_price', 'mean_item_price', 'max_item_price', 'total_freight_value', 'mean_freight_value', 'mean_product_weight_g', 'product_category_count', 'seller_state_count', 'payment_count', 'total_payment_value', 'mean_payment_value', 'max_payment_installments']


In [4]:
required_columns = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

print(ml_table[required_columns].head())

  order_delivered_customer_date order_estimated_delivery_date
0           2017-10-10 21:25:13           2017-10-18 00:00:00
1           2018-08-07 15:27:45           2018-08-13 00:00:00
2           2018-08-17 18:06:29           2018-09-04 00:00:00
3           2017-12-02 00:28:42           2017-12-15 00:00:00
4           2018-02-16 18:17:02           2018-02-26 00:00:00


## 1. Prepare the delivery dates

The delivery date columns are converted to datetime format so that
they can be compared correctly.

In [ ]:
ml_table["order_delivered_customer_date"] = pd.to_datetime(
    ml_table["order_delivered_customer_date"],
    errors="coerce"
)

ml_table["order_estimated_delivery_date"] = pd.to_datetime(
    ml_table["order_estimated_delivery_date"],
    errors="coerce"
)

## 2. Create the delivery label

An order is considered late when the actual customer delivery date
is later than the estimated delivery date.

- `1` = Late
- `0` = On time

In [5]:
ml_table["is_late"] = (
    ml_table["order_delivered_customer_date"]
    > ml_table["order_estimated_delivery_date"]
).astype(int)

In [6]:
print(ml_table["is_late"].value_counts())

is_late
0    91614
1     7827
Name: count, dtype: int64


In [7]:
print(ml_table["is_late"].value_counts(normalize=True))

is_late
0    0.92129
1    0.07871
Name: proportion, dtype: float64


## 3. Label Distribution

The dataset contains 99,441 orders.

- 91,614 orders (92.13%) were delivered on time.
- 7,827 orders (7.87%) were delivered late.

This shows a class imbalance because late deliveries represent only 7.87%
of the dataset. Therefore, accuracy alone should not be used as the main
evaluation metric when training the model.

In [8]:
check = ml_table[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "is_late"
    ]
].sample(10, random_state=42)

check

,order_id,order_delivered_customer_date,order_estimated_delivery_date,is_late
52263,b83d3a21b98b819697f37668a420a43e,2018-05-16 18:52:41,2018-05-29 00:00:00,0
46645,bd65649afc908d9afe43db0bcd7c1de5,2018-04-17 21:08:56,2018-04-30 00:00:00,0
37546,67b50899f52995848c427e361e10dde3,2018-06-27 13:17:27,2018-07-16 00:00:00,0
94756,32733fc014b67ef70fa6039dd8c6ba82,2017-09-25 17:53:23,2017-09-22 00:00:00,1
14771,39a70e9e9b729b11dee34ac12478597f,2017-08-22 16:45:00,2017-09-12 00:00:00,0
36263,80000ae9d118d79953522e35cce34f13,2017-03-08 08:51:57,2017-03-20 00:00:00,0
98556,2bfd14409ba8ba1153ce42b2abc44bb8,2018-01-05 13:46:27,2018-01-18 00:00:00,0
23747,faa01b7a24d0ba51d0d0ec652cd9b745,2018-07-25 11:52:20,2018-08-06 00:00:00,0
50315,0f93c58739ea2f69d93afa96a7a07855,2017-12-07 21:10:00,2017-12-12 00:00:00,0
6501,e389f1491d9b728672f73f3d7be14533,2018-01-08 22:20:36,2018-01-22 00:00:00,0


In [9]:
print("Missing labels:", ml_table["is_late"].isna().sum())

Missing labels: 0


In [10]:
print("Total rows:", len(ml_table))
print("Unique orders:", ml_table["order_id"].nunique())

Total rows: 99441
Unique orders: 99441


## 4. Save the Labeled Dataset

The labeled order-level table is saved as an artifact for the next
notebook. Notebook 3 will use this file to create the train,
validation, and test splits.

In [11]:
import os

os.makedirs("artifacts", exist_ok=True)

ml_table.to_csv(
    "artifacts/labeled_orders.csv",
    index=False
)

print("Labeled artifact saved successfully!")

Labeled artifact saved successfully!


In [12]:
import os

print(os.path.exists("artifacts/labeled_orders.csv"))

True
